In [ ]:
import os
import pandas as pd
from bs4 import BeautifulSoup
import lxml
from io import StringIO

In [ ]:
SCORES_DIR = "data2/scores"
boxscores = os.listdir(SCORES_DIR)

In [ ]:
len(boxscores)
boxscores

In [ ]:
box_scores = [os.path.join(SCORES_DIR, f).replace("\\", "/") for f in boxscores if ".html" in f ]


In [ ]:
soup

In [ ]:
pip install lxml

In [ ]:
teams = list(line_scores["Teams"])
teams

In [ ]:
# for team in teams:

base_cols = None
games = []

def read_stats(soup,team,stat):
    stats = pd.read_html(StringIO(str(soup)),attrs = {"id":f"box-{team}-game-{stat}"},index_col=0)[0]
    stats = stats.apply(pd.to_numeric,errors="coerce")
    return stats

    
def read_season_info(soup):
    nav = soup.select("#bottom_nav_container")[0]
    hrefs = [a["href"] for a in nav.find_all("a")]
    season = os.path.basename(hrefs[1]).split("_")[0]
    return season


def read_line_score(soup):
    line_score = pd.read_html(StringIO(str(soup)),attrs ={"id":"line_score"})[0]
    cols = list(line_score.columns)
    cols[0] = "Teams"
    cols[-1] = "Total"
    line_score.columns = cols
    line_score = line_score[["Teams","Total"]]
    return line_score

def parse_html(boxscore):
    with open(boxscore,"r",encoding="utf-8") as f:
        html = f.read()
    soup = BeautifulSoup(html)
    [s.decompose() for s in soup.select("tr.over_header")]
    [s.decompose() for s in soup.select("tr.thead")]
    return soup



for boxscore in box_scores:
    soup = parse_html(boxscore)
    line_score = read_line_score(soup)
    teams = list(line_score["Teams"])
    summaries=[]
    for team in teams:
        
        basic = read_stats(soup,team,"basic")   
        advanced = read_stats(soup,team,"advanced")
        totals = pd.concat([basic.iloc[-1,:],advanced.iloc[-1,:]])
        totals.index = totals.index.str.lower()
     
        maxes = pd.concat([basic.iloc[:-1,:].max(),advanced.iloc[:-1,:].max()])
        maxes.index = maxes.index.str.lower() + "_max"
        summary = pd.concat([totals,maxes])
     
        if base_cols is None:
             base_cols = list(summary.index.drop_duplicates(keep="first"))
             base_cols = [l for l in base_cols if "bpm" not in l]
        summary = summary[base_cols]
        summaries.append(summary)
    summary = pd.concat(summaries,axis=1).T
   
    summary["home"] = [0,1]
    
    game = pd.concat([summary,line_score],axis=1)
    game_opp = game.iloc[::-1].reset_index()
    game_opp.columns = game_opp.columns + "_opp"
    full_game = pd.concat([game,game_opp],axis=1)
        
    full_game["season"] = read_season_info(soup)
    full_game["date"] = os.path.basename(boxscore)[:8]
    full_game["date"] = pd.to_datetime(full_game["date"],format="%Y%m%d")
    full_game["won"] = full_game["Total"]>full_game["Total_opp"]
    games.append(full_game)
    
    if len(games) % 100 == 0:
        print("hey bro")

    
# basic
# totals
# maxes



In [ ]:
games_df = pd.concat(games, ignore_index=True)
len(games_df)

In [ ]:
games_df.to_csv("games.csv",index=False)

In [ ]:
games